In [230]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [231]:
import numpy as np
import matplotlib.pyplot as plt
import math
import time
from src.model import FiLMResNet2In, flatten_last
from src.normalizer import RunningMeanStd
from src.envpacker import packenv, packenv_batch
from src.utils import transition_ability_batched, update_v_history
import torch
from torch import nn
import torch.nn.functional as F

## 1. Configs

In [232]:
# Training configs
AGENTS = 50     # number of agents
LEARNING_RATE = 1e-3
TRAINING_STEPS = 30000
BATCH_SIZE = 10
DISPLAY_STEP = 1000 # For visualization
TRAIN_STEP_INTERVAL = 2 # Interval of steps between training episodes


# Bewley model parameters
theta = 1 # CRRA
beta = 0.975 # Discount factor
A = 1 # Technology parameter
alpha = 0.33 # Capital share of income
gamma = 2 # Inverse Frisch elasticity
########################################### (MiLF inputs)
r = 0.04 # Interest rate (Return to savings)
w = 1 # Wage rate (Return to labor)
delta = 0.06 # Depreciation rate of capital
TAX_PARAMS = {
    "tax_consumption": 0.065,          # Consumption tax (fixed)
    "tax_income": 0.2,                # Tax on labor income
    "income_tax_elasticity": 0.5,     # Elasticity of labor supply w.r.t. after-tax income
    "saving_tax_elasticity": 0.5,     # Elasticity of savings w.r.t. after-tax income
    "tax_saving": 0.1                 # Tax on interest income
}
###########################################
p = 2.2e-6
q = 0.99

# shock parameters
# log e' = rho_v * log e + sigma_v * epsilon, epsilon ~ N(0,1)
rho_v = 0.95 # persistence of ability shock
sigma_v = 0.2 # std of ability shock
v_bar = 1.5


In [233]:
# Bounds of shock 
v_min = math.exp(-2 * sigma_v  / math.sqrt(1-rho_v**2))
v_max = math.exp( 2 * sigma_v  / math.sqrt(1-rho_v**2))

## 2. Helper functions and classes

In [234]:
def mean_across_agents(x): # Since agent number is fix, thus we can use mean instead of sum
    return torch.mean(x, dim=1, keepdim=True)


def calculate_price(savings, ability, labor):
    savings_aggregate, labor_aggregate_effective = mean_across_agents(savings), mean_across_agents(labor * ability)
    wage = A * (1-alpha) * ((savings_aggregate/labor_aggregate_effective) ** alpha)
    ret = A * alpha * (savings_aggregate/labor_aggregate_effective ** alpha)
    return wage, ret

def taxfunc(ibt, abt, taxparams=TAX_PARAMS):
    it = ibt - (1 - taxparams["tax_income"]) * (ibt**(1-taxparams["income_tax_elasticity"])/(1-taxparams["income_tax_elasticity"])) # individual after tax income
    at = abt - (1-taxparams["tax_saving"]/1-taxparams["saving_tax_elasticity"]) * (abt**(1-taxparams["saving_tax_elasticity"])) # individual after tax saving
    return it, at

def calculate_moneydisposable(wage, ret, ability, labor, savings, delta):
    ibt = wage * labor * ability + (1-delta+ret) * savings   # individual before tax income

    it, at = taxfunc(ibt = ibt, abt=savings)
    money_disposable = it + at

    return money_disposable, ibt


def output_transform(savings, money_disposable):

    # The a here is saving rate coming from the NN output
    consumption = money_disposable * (1 - savings)
    savings = money_disposable * savings    
    return consumption, savings


# def laborfocloss(savings, labor, ibt, money_disposable, wage, ability, taxparams=TAX_PARAMS):

#     loss_foc =  -labor ** (-gamma) + ((1-savings)*money_disposable/(1+taxparams["tax_consumption"])) * \
#         (wage * ability) * (1 - taxparams["tax_income"]) * (ibt ** (-taxparams["income_tax_elasticity"]))

#     return torch.abs(loss_foc)



def fbloss(savings_ratio, mutilpier):
    R1 = savings_ratio 
    R2 = (1-mutilpier) 
    return torch.mean(R1+R2-torch.sqrt(R1**2+R2**2))

# 仔細檢查這個loss

def auxiloss(c0, c1_A, c1_B, savings2_A, savings2_B, mu_t0,
             ibt_A, ibt_B, ret0, theta, taxparams=TAX_PARAMS):

    # --- compute auxiliary terms ---
    aux_1 = beta * (c1_A / c0) ** (-theta) * \
        (ret0 * (1 - taxparams["tax_income"]) * (ibt_A) ** taxparams["income_tax_elasticity"]) + \
        (1 - taxparams["tax_saving"]) * savings2_A ** taxparams["saving_tax_elasticity"] - \
        mu_t0

    aux_2 = beta * (c1_B / c0) ** (-theta) * \
        (ret0 * (1 - taxparams["tax_income"]) * (ibt_B) ** taxparams["income_tax_elasticity"]) + \
        (1 - taxparams["tax_saving"]) * savings2_B ** taxparams["saving_tax_elasticity"] - \
        mu_t0

    # --- NaN / Inf / shape diagnostics ---
    def check_tensor(name, t):
        print(f"\n{name}:")
        if isinstance(t, torch.Tensor):
            print(f"  shape: {tuple(t.shape)}")
            print(f"  min/max: {t.min().item():.4e} / {t.max().item():.4e}")
            print(f"  mean: {t.mean().item():.4e}")
            print(f"  has_nan: {torch.isnan(t).any().item()}  has_inf: {torch.isinf(t).any().item()}")
        elif isinstance(t, (float, int)):
            print(f"  scalar value: {t:.6e}")
        else:
            print(f"  type: {type(t)} — not a tensor or float")

    check_tensor("c0", c0)
    check_tensor("c1_A", c1_A)
    check_tensor("c1_B", c1_B)
    check_tensor("savings2_A", savings2_A)
    check_tensor("savings2_B", savings2_B)
    check_tensor("ibt_A", ibt_A)
    check_tensor("ibt_B", ibt_B)
    check_tensor("ret0", ret0)
    check_tensor("mu_t0", mu_t0)
    check_tensor("aux_1", aux_1)
    check_tensor("aux_2", aux_2)

    # --- debug NaN propagation ---
    if torch.isnan(aux_1).any() or torch.isnan(aux_2).any():
        print("\n⚠️ NaN detected in aux_1 or aux_2!")
        with torch.no_grad():
            print("aux_1 example:", aux_1[torch.isnan(aux_1)][:5])
            print("aux_2 example:", aux_2[torch.isnan(aux_2)][:5])

    loss = torch.mean(aux_1 * aux_2)
    print(f"\nloss: {loss.item() if torch.isfinite(loss) else 'NaN/Inf'}")
    return loss

# def auxiloss(c0, c1_A, c1_B, savings2_A, savings2_B, mu_t0,
#              ibt_A, ibt_B, ret0, theta, taxparmas = TAX_PARAMS):

#     aux_1 = beta * (c1_A / c0) ** (-theta) * \
#         (ret0 * (1 - taxparmas["tax_income"]) * (ibt_A) ** taxparmas["income_tax_elasticity"]) + \
#         (1 - taxparmas["tax_saving"]) * savings2_A ** taxparmas["saving_tax_elasticity"] - \
#         mu_t0

#     aux_2 = beta * (c1_B / c0) ** (-theta) * \
#         (ret0 * (1 - taxparmas["tax_income"]) * (ibt_B) ** taxparmas["income_tax_elasticity"]) + \
#         (1 - taxparmas["tax_saving"]) * savings2_B ** taxparmas["saving_tax_elasticity"] - \
#         mu_t0

#     print(f"aux_1 : {aux_1} / aux_2 : {aux_2}")

#     return torch.mean(aux_1*aux_2)

def laborfocloss(labor, consumption, ibt, wage, ability, taxparams=TAX_PARAMS):
    loss_foc = -labor**gamma + \
        ((consumption**-(theta)) / (1+ taxparams["tax_saving"])) * \
            (wage*ability*(1-taxparams["tax_income"]*ibt**(-taxparams["income_tax_elasticity"])))
    return torch.mean(torch.abs(loss_foc))



In [235]:
state_dim = 2*AGENTS + 2 # two state variables for each agent + 2 individual variables
cond_dim = 5 # exogenous variables for all agents in all worlds(Batch)
model = FiLMResNet2In(state_dim=state_dim, cond_dim=cond_dim,
                        hidden_dim=128, num_res_blocks=3, output_dim=3, dropout=0.1)

In [236]:
def initial_state(required_batch_size, tax_params_dict=TAX_PARAMS):
    # 隨機產生初始資產與儲蓄
    moneydisposable = np.random.lognormal(0.1, 2.0, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)
    savings = np.random.lognormal(0.1, 2.0, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)

    # Initial productivity
    ability = np.random.lognormal(v_min, v_max, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)
    ability = ability / np.mean(ability, axis=1, keepdims=True)


    # superstar 標誌 (v1 對應一組, v2 對應一組)
    is_superstar_v1 = np.zeros((required_batch_size, AGENTS), dtype=bool)
    is_superstar_v2 = np.zeros((required_batch_size, AGENTS), dtype=bool)

    # 稅制參數轉為 tensor
    tax_params = torch.tensor(list(tax_params_dict.values()), dtype=torch.float32)
    tax_params = tax_params.repeat(required_batch_size, 1)

    # 轉為 tensor
    moneydisposable_t = torch.tensor(moneydisposable, dtype=torch.float32)
    savings_t = torch.tensor(savings, dtype=torch.float32)
    ability_t = torch.tensor(ability, dtype=torch.float32)
    is_superstar_vA = torch.tensor(is_superstar_v1, dtype=torch.bool)
    is_superstar_vB = torch.tensor(is_superstar_v2, dtype=torch.bool)
    tax_params_t = tax_params

    # 回傳字典
    return {
        "moneydisposable": moneydisposable_t,
        "savings":savings_t,
        "ability":ability_t,
        "is_superstar_vA": is_superstar_vA,
        "is_superstar_vB": is_superstar_vB,
        "tax_params": tax_params_t
    }


In [237]:
def build_inputs(moneydisposable, v, tax_params):
    """
    回傳:
      features : (B, A, 2A + 2)          # 給模型輸入
      condi    : (B, A, Z)              # 稅制條件
      env_info : dict                   # 僅供環境轉移使用，不進模型
    """
    B, A = moneydisposable.shape

    # (B, Z) -> (B, A, Z)
    condi = tax_params.unsqueeze(1).expand(-1, A, -1)

    # (B, 2A) -> (B, A, 2A)
    sum_info = torch.cat([moneydisposable, v], dim=1)         # (B, 2A)
    sum_info_rep = sum_info.unsqueeze(1).expand(-1, A, -1)    # (B, A, 2A)

    # (B, A, 1) × 2
    money_self = moneydisposable.unsqueeze(-1)  # (B, A, 1)
    v_self     = v.unsqueeze(-1)                # (B, A, 1)

    # 給模型的 features
    features = torch.cat([sum_info_rep, money_self, v_self], dim=2)  # (B, A, 2A+2)

    return features, condi


In [238]:
def run_network_given_state(state, model, brach=None):
    
    # 把整個state agent 需要作動的部分取出來

    if not brach:
        agent_state = build_inputs(
            moneydisposable=state["moneydisposable"],
            v=state["ability"],
            tax_params=state["tax_params"]
        )

    else:
        # print(state["moneydisposable"])
        agent_state = build_inputs(
            moneydisposable=state["moneydisposable"],
            v=state[f"ability_{brach}"],
            tax_params=state["tax_params"]
        )

    acts = [torch.sigmoid, lambda x: F.softplus(x) + 1e-6, torch.sigmoid]
    out = model(agent_state[0], agent_state[1])
    savings_t1, mu_t0, labor_t0 = [acts[i](out[..., i]) for i in range(out.shape[-1])]
    savings_t1, mu_t0, labor_t0 = savings_t1.squeeze(-1), mu_t0.squeeze(-1), labor_t0.squeeze(-1)

    return {
        "savings_t1": savings_t1,
        "mu_t0": mu_t0,
        "labor_t0": labor_t0,
    }



In [239]:
def part_transition_transform(state, model_out, branch=None, is_init=False):
    

    # Current wage and return
    wage, ret = calculate_price(
        savings=state["savings"],
        ability=state["ability"] if not branch else state[f"ability_{branch}"],
        labor=model_out["labor_t0"],
    )

    # if is init, interest rate comes from the previous period
    money_disposable, ibt = calculate_moneydisposable(wage=wage, ret=ret, 
                                                ability=state["ability"]if not branch else state[f"ability_{branch}"],
                                                labor=model_out["labor_t0"], 
                                                savings=r if is_init else state["savings"],
                                                delta=delta)
    
    consumption, savings = output_transform(savings=model_out["savings_t1"], money_disposable=money_disposable)

    return {
        "moneydisposable": money_disposable,
        "savings": savings, # savings for next period
        "consumption": consumption, # consumption for current period
        "wage": wage,
        "ret": ret,
        "ibt": ibt,
        "tax_params": state["tax_params"]
    }

In [240]:
# 在這裡加入shock
# def update_state_dict(part_transition_output, state_dicta):
#     state_dict["moneydisposable"] = part_transition_output["moneydisposable"]
#     state_dict["savings"] = part_transition_output["savings"]
#     state_dict[]

In [241]:
def run_model(state_dict_train, model, v_history_A, v_history_B):

    state_dict_tmp = {}
    # t=0
    model_out_0 = run_network_given_state(state_dict_train, model=model)

    # moneydisposable, savings, consumption, wage, ret, ibt
    part_transition_output = part_transition_transform(state=state_dict_train, model_out=model_out_0, is_init=True)
    # keys_to_keep = ["moneydisposable", "savings", "tax_params", "consu"]
    keys_to_keep = ["moneydisposable", "savings", "consumption", "wage", "ret", "ibt", "tax_params"]
    state_dict_tmp = {k: part_transition_output[k] for k in keys_to_keep if k in part_transition_output}

    v_next_A, is_superstar_next_A = transition_ability_batched(
        ability=state_dict_train["ability"],
        is_superstar_prev=state_dict_train["is_superstar_vA"],
        v_history=v_history_A,
        rho_v=rho_v, sigma_v=sigma_v,
        p=p, q=q, v_bar=v_bar,
        v_min=v_min, v_max=v_max
    )

    v_next_B, is_superstar_next_B = transition_ability_batched(
        ability=state_dict_train["ability"],
        is_superstar_prev=state_dict_train["is_superstar_vB"],
        v_history=v_history_A,
        rho_v=rho_v, sigma_v=sigma_v,
        p=p, q=q, v_bar=v_bar,
        v_min=v_min, v_max=v_max
    )

    v_history_A = update_v_history(
        v_history=v_history_A,
        v_next=v_next_A
    )
    v_history_B = update_v_history(
        v_history=v_history_B,
        v_next=v_next_B
    )

    state_dict_tmp["ability_A"] = v_next_A
    state_dict_tmp["ability_B"] = v_next_B


    # t=1
    model_out_1A = run_network_given_state(state_dict_tmp, model=model, brach="A")
    model_out_1B = run_network_given_state(state_dict_tmp, model=model, brach="B")
    return model_out_1A

    # part_transition_output_A = part_transition_transform(state_dict_tmp, model_out_1A, branch="A", is_init=False)
    # part_transition_output_B = part_transition_transform(state_dict_tmp, model_out_1B, branch="B", is_init=False)



    # # --- calculate loss ---
    # fb_loss = fbloss(savings_ratio=model_out_0["savings_t1"], mutilpier=model_out_0["mu_t0"])
    # aux_loss = auxiloss(c0=state_dict_tmp["consumption"],
    #          c1_A=part_transition_output_A["consumption"],c1_B=part_transition_output_B["consumption"],
    #          mu_t0=model_out_0["mu_t0"],
    #          savings2_A=part_transition_output_A["savings"], savings2_B=part_transition_output_A["savings"],
    #          ibt_A=part_transition_output_A["ibt"], ibt_B=part_transition_output_B["ibt"],
    #          ret0=r, theta=theta)
    # laborfoc_loss = laborfocloss(labor=model_out_0["labor_t0"],
    #              consumption=state_dict_tmp["consumption"],
    #              ibt = state_dict_tmp["ibt"],
    #              wage=state_dict_tmp["wage"],
    #              ability=state_dict_train["ability"],
    #              taxparams=TAX_PARAMS)
    # # print(fb_loss, aux_loss, laborfoc_loss)

    # state_dict_train["is_superstar_vA"] = is_superstar_next_A 
    # state_dict_train["is_superstar_vB"] = is_superstar_next_B

    # return state_dict_train, v_history_A, v_history_B, fb_loss+aux_loss+laborfoc_loss

In [242]:
# run_model(state_dict_train, model, v_history_A=v_history_A, v_history_B=v_history_B)

In [244]:
state_dict_train = initial_state(required_batch_size=256)
v_history_A = None
v_history_B = None

test = run_model(state_dict_train = state_dict_train, 
                                                               model = model, 
                                                               v_history_A=v_history_A, 
                                                               v_history_B=v_history_B)

In [245]:
test

{'savings_t1': tensor([[0.4019, 0.4204, 0.4323,  ..., 0.3835, 0.4356, 0.4027],
         [0.4969, 0.4803, 0.4555,  ..., 0.4574, 0.4687, 0.4565],
         [0.4775, 0.4266, 0.4545,  ..., 0.4796, 0.4339, 0.4614],
         ...,
         [0.3649, 0.3529, 0.3670,  ..., 0.3373, 0.3815, 0.3906],
         [0.4402, 0.4394, 0.4596,  ..., 0.4551, 0.4284, 0.4147],
         [0.4983, 0.4677, 0.4936,  ..., 0.4848, 0.4952, 0.4538]],
        grad_fn=<SqueezeBackward1>),
 'mu_t0': tensor([[0.7251, 0.9275, 0.9281,  ..., 0.7725, 0.8453, 0.8493],
         [0.7097, 0.8671, 0.7543,  ..., 0.7491, 0.7725, 0.7274],
         [0.7234, 0.6045, 0.5957,  ..., 0.7181, 0.6089, 0.5884],
         ...,
         [0.7254, 0.8158, 0.8094,  ..., 0.7642, 0.8273, 0.6929],
         [0.7873, 0.8179, 0.7619,  ..., 0.8631, 0.8090, 0.7558],
         [0.7844, 0.7258, 0.7175,  ..., 0.7736, 0.8701, 0.6627]],
        grad_fn=<SqueezeBackward1>),
 'labor_t0': tensor([[0.4883, 0.4808, 0.5002,  ..., 0.4808, 0.4911, 0.5078],
         [0.5613

In [243]:
state_dict_train = initial_state(required_batch_size=256)
v_history_A = None
v_history_B = None

for i in range(5):
    if i % TRAIN_STEP_INTERVAL == 0:
        
        state_dict_train, v_history_A, v_history_B, loss = run_model(state_dict_train = state_dict_train, 
                                                               model = model, 
                                                               v_history_A=v_history_A, 
                                                               v_history_B=v_history_B)
        # print(f"Training stpe {i}, loss : {loss}")
        break

ValueError: not enough values to unpack (expected 4, got 3)

In [17]:
v_history_A.shape

torch.Size([3, 256, 50])